In [2]:
!pip -q install chromadb sentence-transformers transformers pypdf torch

import re
import torch
import chromadb
from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

CHUNK_SIZE = 600
CHUNK_OVERLAP = 120
TOP_K = 5

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)

print("Loading LLM...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1
)

print("Loading ChromaDB...")
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="pdf_collection"
)

print("READY")


def extract_pages(file):
    reader = PdfReader(file)
    pages = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()

        if text:
            text = re.sub(r"\s+", " ", text).strip()

            if text:
                pages.append({
                    "page": i + 1,
                    "text": text
                })

    return pages


def chunk_pages(pages):
    chunks = []

    for p in pages:
        text = p["text"]
        page = p["page"]

        start = 0

        while start < len(text):
            end = start + CHUNK_SIZE
            chunk = text[start:end]

            chunks.append({
                "text": chunk,
                "page": page
            })

            start += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks


def build_index(chunks):
    global collection

    try:
        chroma_client.delete_collection("pdf_collection")
    except:
        pass

    collection = chroma_client.get_or_create_collection(
        name="pdf_collection"
    )

    texts = [c["text"] for c in chunks]

    embeddings = embedder.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        batch_size=32
    ).tolist()

    ids = [f"chunk_{i}" for i in range(len(chunks))]
    metadatas = [{"page": c["page"]} for c in chunks]

    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas
    )


def retrieve(query):
    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=TOP_K
    )

    retrieved = []

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    for doc, meta in zip(docs, metas):
        retrieved.append({
            "text": doc,
            "page": meta["page"]
        })

    return retrieved


def build_context(chunks):
    context = ""

    for i, c in enumerate(chunks):
        context += f"""
[Source {i+1} | Page {c['page']}]
{c['text']}

"""

    return context


def ask_llm(context, query):
    prompt = f"""
You are a strict document-based question answering assistant.

Rules:
- Answer only from the provided context.
- Do not invent information.
- If the answer is not found, say:
  "I could not find the answer in the uploaded document."
- Mention page numbers when possible.

Context:
{context}

Question:
{query}

Answer:
"""

    output = generator(
        prompt,
        max_new_tokens=300,
        temperature=0.0,
        do_sample=False,
        return_full_text=False
    )

    return output[0]["generated_text"].strip()


def upload_and_index():
    uploaded = files.upload()

    if not uploaded:
        return

    pdf_path = list(uploaded.keys())[0]

    print("Reading PDF...")
    pages = extract_pages(pdf_path)

    print("Chunking...")
    chunks = chunk_pages(pages)

    print("Building index...")
    build_index(chunks)

    print("READY")

    return chunks


def ask_loop():
    while True:
        q = input("\nAsk a question (type 'exit' to quit): ")

        if q.lower() == "exit":
            break

        retrieved = retrieve(q)
        context = build_context(retrieved)
        answer = ask_llm(context, q)

        print("\nAnswer:\n")
        print(answer)


upload_and_index()
ask_loop()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading LLM...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

...


Answer:

Student's ID is 21101348. This can be seen in both Source 1 and Source 2, as well as in Source 3. However, it is mentioned in Source 4 under the 'Thesis Coordinator' section, indicating that the student's ID was used in the thesis coordinator's role. Therefore, I cannot definitively state that the student's ID is 21101348 based solely on these sources. The most reliable source would be Source 4 if we were to consider it as the definitive reference point. 

The final answer is: I could not find the answer in the uploaded document. 
(Referencing the above reasoning process.)

Ask a question (type 'exit' to quit): Declaration


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:

The declaration states that the thesis submitted is the author's own original work completed at Brac University. It also mentions that no material from previous publications or submissions was used, and it follows proper citation guidelines. Additionally, the thesis does not include any material that has been accepted or submitted elsewhere for any degree or diploma. The declaration includes acknowledgments for all main sources of help received during the research process. 

Page Numbers: 
- Source 1: Page 2
- Source 2: Page 7
- Source 3: Page 23
- Source 4: Page 2
- Source 5: Page 3

I could not find the answer in the uploaded document.

Ask a question (type 'exit' to quit): Methodology in Brief


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:

This paper is mainly focused on my works in the company. I was given a lot of training and gained technical expertise, then I was given some tasks. I also learned about office culture and etiquette. I also learned a few things for my own growth as a programmer. (Page 7)

Ask a question (type 'exit' to quit): exit
